# Module 7: Portfolio Project Starter

## Learning Objectives
By completing this notebook you will:
1. Have a working skeleton for a complete demand forecasting system
2. Run the reference implementation on French Bakery data end-to-end
3. Know exactly which sections to adapt for your own dataset and business question

## Prerequisites
- Module 01: NHITS training and `.fit()` / `.predict()` API
- Module 02: `MQLoss`, CRPS evaluation, calibration checking
- Module 03: `.simulate()` and the Monte Carlo business framework
- Module 04: `.explain()` and attribution interpretation

## Estimated Time: 15 minutes (reference run) + your project time

---

**How to use this notebook:**

1. Run Section 0 through Section 5 as-is. This is the complete reference implementation on French Bakery data.
2. Confirm every section produces output without errors.
3. Replace Section 1 (data loading) with your chosen dataset.
4. Update the business question in Section 4 to match your domain.
5. Re-run all sections.

Sections marked `[ADAPT]` are the ones you will modify.

In [ ]:
learning_objectives([
    "Module 01: NHITS training and `.fit()` / `.predict()` API",
    "Module 02: `MQLoss`, CRPS evaluation, calibration checking",
    "Module 03: `.simulate()` and the Monte Carlo business framework",
    "Module 04: `.explain()` and attribution interpretation",
    "**How to use this notebook:**"
])

## Section 0: Environment Setup

Install and import all required packages. Run this cell once.

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
# Install required packages (uncomment if running for the first time)
# !pip install neuralforecast datasetsforecast utilsforecast statsforecast matplotlib pandas numpy

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MQLoss

from utilsforecast.evaluation import evaluate
from utilsforecast.losses import crps, coverage

from statsforecast import StatsForecast
from statsforecast.models import SeasonalNaive

np.random.seed(42)
print("Environment ready.")

## Section 1: Data Loading [ADAPT]

Load your chosen dataset in nixtla format: three columns `unique_id`, `ds`, `y`.

The reference implementation below uses **French Bakery daily sales** — 8 bakery products with strong weekly seasonality.
Replace this section with your own dataset for the actual project.

**Other dataset options:**
```python
# M5 (Walmart retail) — use a subset of 10 series to start
from datasetsforecast.m5 import M5
df_all, *_ = M5.load("./data/")
series_sample = df_all["unique_id"].unique()[:10]
df = df_all[df_all["unique_id"].isin(series_sample)].copy()

# ETT Energy (hourly load)
from datasetsforecast.long_horizon import LongHorizon
df, *_ = LongHorizon.load("./data/", group="ETTh1")

# Australian Tourism (quarterly)
from datasetsforecast.long_horizon import LongHorizon
df, *_ = LongHorizon.load("./data/", group="Tourism")
```

In [ ]:
# ── Reference implementation: French Bakery ──────────────────────────────────
# Replace this block with your chosen dataset.

BAKERY_URL = (
    "https://raw.githubusercontent.com/Nixtla/transfer-learning-time-series/"
    "main/datasets/french_bakery_daily.csv"
)

df = pd.read_csv(BAKERY_URL, parse_dates=["ds"])

# ── Validate nixtla format ────────────────────────────────────────────────────
# This check applies to any dataset — keep it.
required_cols = {"unique_id", "ds", "y"}
assert required_cols.issubset(df.columns), (
    f"Missing columns: {required_cols - set(df.columns)}. "
    "Rename your columns to unique_id, ds, y."
)
assert pd.api.types.is_datetime64_any_dtype(df["ds"]), (
    "Column 'ds' must be datetime. Use pd.to_datetime() or parse_dates=['ds']."
)

print(f"Series count:  {df['unique_id'].nunique()}")
print(f"Date range:    {df['ds'].min().date()} to {df['ds'].max().date()}")
print(f"Total rows:    {len(df):,}")
print(f"Series names:  {df['unique_id'].unique().tolist()}")
df.head()

## Section 2: Exploratory Data Analysis

Plot the series and identify the dominant seasonal period. This step does not change between
the reference implementation and your project — just run it on your data.

In [ ]:
# Plot up to 4 series to assess seasonal structure
series_to_plot = df["unique_id"].unique()[:4]
fig, axes = plt.subplots(len(series_to_plot), 1, figsize=(14, 3 * len(series_to_plot)), sharex=True)
if len(series_to_plot) == 1:
    axes = [axes]

for ax, sid in zip(axes, series_to_plot):
    subset = df[df["unique_id"] == sid].sort_values("ds")
    ax.plot(subset["ds"], subset["y"], linewidth=0.8, color="steelblue")
    ax.set_title(str(sid), fontsize=10, loc="left")
    ax.set_ylabel("y")
    ax.xaxis.set_major_locator(ticker.MaxNLocator(8))

plt.suptitle("Time Series Overview", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("eda_overview.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: eda_overview.png")

In [ ]:
# ── Train / test split ────────────────────────────────────────────────────────
# Forecast horizon h: set to match your business question.
# For daily bakery data, 7 days = one week of inventory planning.
# [ADAPT] Change h to match the planning horizon in your business question.

H = 7  # forecast horizon in time steps

# Hold out the last H steps of each series as the test set
# This is the standard evaluation setup for time series
test_mask = df.groupby("unique_id")["ds"].transform(
    lambda x: x >= x.nlargest(H).min()
)
df_train = df[~test_mask].copy()
df_test  = df[test_mask].copy()

print(f"Train rows: {len(df_train):,}")
print(f"Test rows:  {len(df_test):,}  (last {H} steps per series)")

## Section 3: Model Training and Evaluation [ADAPT]

Train NHITS with `MQLoss` to produce probabilistic forecasts. Verify calibration before moving
to sample paths — a miscalibrated model produces unreliable business answers.

**When to adapt:**
- Change `h` to match your horizon
- Change `input_size` to at least `4 * h` (4x lookback is a good default)
- Change `freq` to match your data frequency (`'D'` daily, `'M'` monthly, `'H'` hourly)
- Consider `XLinear` as an alternative if training is slow on your dataset

```python
# XLinear alternative: faster, surprisingly strong on many datasets
from neuralforecast.models import XLinear
model = XLinear(h=H, input_size=4*H, loss=MQLoss(level=[80, 90]))
```

In [ ]:
# ── Model definition ──────────────────────────────────────────────────────────
# [ADAPT] Adjust h, input_size, and freq to match your dataset.

model = NHITS(
    h=H,
    input_size=4 * H,        # 4x lookback — standard starting point
    loss=MQLoss(level=[80, 90]),
    scaler_type="robust",    # essential for real-world data with outliers
    max_steps=1000,
    val_check_steps=50,
    early_stop_patience_steps=5,
)

nf = NeuralForecast(models=[model], freq="D")  # [ADAPT] change freq if needed
nf.fit(df=df_train)
print("Training complete.")

In [ ]:
# Generate forecasts on the test set
# predict() forecasts the next H steps after the end of df_train
forecasts = nf.predict()
print(f"Forecast columns: {forecasts.columns.tolist()}")
forecasts.head()

In [ ]:
# ── Seasonal naive baseline ───────────────────────────────────────────────────
# Always compare neural models against a classical baseline.

sf = StatsForecast(
    models=[SeasonalNaive(season_length=7)],
    freq="D",
)
sf.fit(df=df_train)
baseline = sf.predict(h=H)

# ── CRPS comparison ───────────────────────────────────────────────────────────
# Lower CRPS = better probabilistic forecast
# For the baseline, we use the point forecast as a degenerate distribution
# (CRPS reduces to MAE for point forecasts)
neural_crps_scores = evaluate(
    df=forecasts.merge(df_test[["unique_id", "ds", "y"]], on=["unique_id", "ds"]),
    metrics=[crps],
    models=["NHITS"],
)

print("CRPS scores (lower is better):")
print(neural_crps_scores.to_string(index=False))

In [ ]:
# ── Calibration check ─────────────────────────────────────────────────────────
# Target: 80% intervals should cover actuals 75%–85% of the time.
# Target: 90% intervals should cover actuals 87%–93% of the time.

forecasts_with_actuals = forecasts.merge(
    df_test[["unique_id", "ds", "y"]], on=["unique_id", "ds"]
)

calibration = evaluate(
    df=forecasts_with_actuals,
    metrics=[coverage],
    models=["NHITS"],
    level=[80, 90],
)

print("Calibration (target: 80% level → 75-85%, 90% level → 87-93%):")
print(calibration.to_string(index=False))

# Milestone 2 check
# If 80% coverage is outside 75-85%, revisit max_steps or scaler_type

## Section 4: Sample Paths and Business Decision [ADAPT]

Generate sample paths using `.simulate()` and apply the Monte Carlo framework to answer
your business question.

**The reference question** (French Bakery):
> What is the cost-optimal weekly stocking level per product, given that stockouts cost
> €12/unit and overstock costs €2/unit?

**[ADAPT] Replace the business question and cost function with your own.**

Template business questions by domain:
- **Inventory**: `optimal_stock = np.quantile(weekly_totals, tau)` where `tau = cost_under / (cost_under + cost_over)`
- **Capacity**: `min_capacity = np.quantile(peak_demand_per_path, 0.90)`
- **Budget reserve**: `reserve_needed = np.quantile(shortfall_per_path, 0.90)`
- **Grid reliability**: `P(supply < demand) = (min_supply_path < demand_path).mean()`

In [ ]:
# ── Generate sample paths ─────────────────────────────────────────────────────
# .simulate() requires the model to have been trained with MQLoss.
# n_paths=200 gives stable Monte Carlo estimates for most business questions.

N_PATHS = 200

paths_df = nf.models[0].simulate(
    futr_df=None,
    step_size=1,
    n_paths=N_PATHS,
)

print(f"paths_df shape: {paths_df.shape}")
print(f"Columns: {paths_df.columns[:5].tolist()} ... {paths_df.columns[-3:].tolist()}")
paths_df.head()

In [ ]:
# ── Visualization 1: Spaghetti plot ───────────────────────────────────────────
# Shows individual sample paths — each line is one plausible future trajectory.

SERIES_TO_PLOT = df["unique_id"].unique()[0]  # [ADAPT] change to your series of interest

series_paths = paths_df[paths_df["unique_id"] == SERIES_TO_PLOT]
path_cols = [c for c in series_paths.columns if c.startswith("sample_")]
path_matrix = series_paths[path_cols].values  # shape: (H, N_PATHS)

# Historical context
history = df_train[df_train["unique_id"] == SERIES_TO_PLOT].sort_values("ds")
history_tail = history.tail(21)  # last 3 weeks for context

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Spaghetti plot (50 paths for readability)
ax = axes[0]
forecast_dates = series_paths["ds"].values
for i in range(min(50, N_PATHS)):
    ax.plot(forecast_dates, path_matrix[:, i], color="steelblue", alpha=0.12, linewidth=0.8)
ax.plot(
    history_tail["ds"], history_tail["y"],
    color="black", linewidth=1.5, label="Historical"
)
ax.plot(
    forecast_dates, np.median(path_matrix, axis=1),
    color="navy", linewidth=2, label="Median"
)
ax.set_title(f"Sample Paths — {SERIES_TO_PLOT}", fontsize=11)
ax.set_ylabel("Demand")
ax.legend()

# Fan chart (80% band)
ax = axes[1]
ax.plot(
    history_tail["ds"], history_tail["y"],
    color="black", linewidth=1.5, label="Historical"
)
ax.plot(
    forecast_dates, np.median(path_matrix, axis=1),
    color="navy", linewidth=2, label="Median"
)
ax.fill_between(
    forecast_dates,
    np.quantile(path_matrix, 0.10, axis=1),
    np.quantile(path_matrix, 0.90, axis=1),
    alpha=0.30, color="steelblue", label="80% band"
)
ax.set_title(f"Fan Chart — {SERIES_TO_PLOT}", fontsize=11)
ax.set_ylabel("Demand")
ax.legend()

plt.tight_layout()
plt.savefig("sample_paths_visualization.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: sample_paths_visualization.png")

In [ ]:
# ── Business Decision: Cost-Optimal Inventory ─────────────────────────────────
# [ADAPT] Replace this section with your own business question.
#
# Reference: newsvendor solution for perishable inventory.
# The optimal stocking level Q* satisfies:
#   Q* = F^{-1}(tau)  where tau = c_under / (c_under + c_over)
# F^{-1} is the quantile function of the weekly total demand distribution.

COST_UNDERSTOCK = 12.0   # [ADAPT] cost per unit of unmet demand (€, $, etc.)
COST_OVERSTOCK  = 2.0    # [ADAPT] cost per unit of unsold inventory

# Critical ratio (newsvendor optimal quantile)
tau = COST_UNDERSTOCK / (COST_UNDERSTOCK + COST_OVERSTOCK)
print(f"Critical ratio tau = {tau:.3f}  (stock at the {tau*100:.1f}th percentile of demand)")

results = {}
for sid in df["unique_id"].unique():
    series_paths_sid = paths_df[paths_df["unique_id"] == sid]
    path_cols_sid = [c for c in series_paths_sid.columns if c.startswith("sample_")]
    pm = series_paths_sid[path_cols_sid].values  # shape: (H, N_PATHS)

    # Weekly total demand across all H steps, per path
    weekly_totals = pm.sum(axis=0)  # shape: (N_PATHS,)

    # Optimal stocking level
    optimal_stock = float(np.quantile(weekly_totals, tau))

    # Stockout probability at this level
    p_stockout = float((weekly_totals > optimal_stock).mean())

    results[sid] = {
        "optimal_weekly_stock": round(optimal_stock, 1),
        "stockout_probability": round(p_stockout, 3),
        "median_weekly_demand": round(float(np.median(weekly_totals)), 1),
        "p80_weekly_demand":    round(float(np.quantile(weekly_totals, 0.80)), 1),
    }

results_df = pd.DataFrame(results).T
results_df.index.name = "product"
print("\nBusiness Decision: Cost-Optimal Weekly Stocking Levels")
print(f"Costs: €{COST_UNDERSTOCK}/unit stockout, €{COST_OVERSTOCK}/unit overstock")
print()
print(results_df.to_string())

## Section 5: Explainability Report [ADAPT]

Run `.explain()` to extract input attributions. Report the top-5 features and interpret
the most important one in domain language.

**What `.explain()` returns:**
- `attributions`: shape `(n_series, input_size)` — Shapley-like feature importance per series
- `feature_names`: list of lag labels, e.g. `['lag_1', 'lag_2', ..., 'lag_28']`

**[ADAPT]** The interpretation sentence in the final cell — translate the top feature
into your domain's language.

In [ ]:
# ── Generate explanations ─────────────────────────────────────────────────────
# .explain() runs a perturbation-based attribution method on the test set.
# This can take 1-2 minutes depending on dataset size.

explanations = nf.models[0].explain(df=df_test, level=90)

print(f"Attribution matrix shape: {explanations['attributions'].shape}")
print(f"Number of features:       {len(explanations['feature_names'])}")
print(f"Feature names (first 5):  {explanations['feature_names'][:5]}")

In [ ]:
# ── Top-5 features by mean absolute attribution ───────────────────────────────

mean_abs_attr = np.abs(explanations["attributions"]).mean(axis=0)
feature_names = explanations["feature_names"]

# Sort descending
sorted_idx = np.argsort(mean_abs_attr)[::-1]
top5_idx   = sorted_idx[:5]

print("Top-5 input features by mean absolute attribution:")
print(f"{'Rank':<6} {'Feature':<15} {'Mean |Attribution|':<20} {'Share %':<10}")
print("-" * 55)
total_attr = mean_abs_attr.sum()
for rank, idx in enumerate(top5_idx, 1):
    share = mean_abs_attr[idx] / total_attr * 100
    print(f"{rank:<6} {feature_names[idx]:<15} {mean_abs_attr[idx]:<20.4f} {share:<10.1f}%")

# Attribution bar chart
fig, ax = plt.subplots(figsize=(8, 4))
top_n = 10
top_idx = sorted_idx[:top_n]
ax.barh(
    [feature_names[i] for i in reversed(top_idx)],
    [mean_abs_attr[i] for i in reversed(top_idx)],
    color="steelblue", edgecolor="white"
)
ax.set_xlabel("Mean Absolute Attribution")
ax.set_title("Input Feature Importance (top 10)", fontweight="bold")
plt.tight_layout()
plt.savefig("explainability_report.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: explainability_report.png")

In [ ]:
# ── Interpretation ────────────────────────────────────────────────────────────
# [ADAPT] Write one sentence translating the top feature into domain language.
# Example below is for the French Bakery dataset.

top_feature   = feature_names[top5_idx[0]]
top_share_pct = mean_abs_attr[top5_idx[0]] / total_attr * 100

print(f"Most important feature: {top_feature}  ({top_share_pct:.1f}% of total attribution)")
print()

# [ADAPT] Replace this sentence with your own domain interpretation.
print(
    f"Interpretation: '{top_feature}' (sales from one week ago on the same day) "
    f"accounts for {top_share_pct:.1f}% of the model's forecast signal, "
    "confirming that weekly regulars — customers who purchase the same product "
    "every week — are the primary driver of bakery demand."
)

## Section 6: Stakeholder Summary

The final deliverable is a single document readable by a non-technical stakeholder.
This cell assembles the executive summary from the numeric results computed above.

**[ADAPT]** Replace the text in the template with your own dataset, business question, and results.

In [ ]:
# ── Executive summary ─────────────────────────────────────────────────────────
# [ADAPT] Customize this template for your project.

print("=" * 70)
print("STAKEHOLDER SUMMARY")
print("=" * 70)
print()

# ── Business Question & Answer ────────────────────────────────────────────────
print("BUSINESS QUESTION")
print("-" * 40)
# [ADAPT] State your business question.
print(
    "What is the cost-optimal weekly stocking level for each bakery product, "
    "given that stockouts cost €12/unit and unsold stock costs €2/unit?"
)
print()
print("ANSWER")
print("-" * 40)
for sid, row in results_df.iterrows():
    print(f"  {sid:<30} {row['optimal_weekly_stock']:>8.0f} units/week")
print()
print(f"  Confidence level: {tau*100:.0f}%  (cost-optimal under given cost ratio)")
print()

# ── Model Performance ─────────────────────────────────────────────────────────
print("MODEL PERFORMANCE")
print("-" * 40)
print(f"  Model:                NHITS with MQLoss (probabilistic)")
print(f"  Calibration check:    see calibration table above")
print(f"  Sample paths:         {N_PATHS} paths over {H}-step horizon")
print()

# ── Key Finding from Explainability ───────────────────────────────────────────
print("KEY FINDING")
print("-" * 40)
# [ADAPT] Insert your interpretation sentence here.
print(
    f"  The strongest predictor of next week's demand is the same day last week "
    f"({top_feature}), accounting for {top_share_pct:.0f}% of forecast variance. "
    "Weekly purchasing habits dominate over trend or promotional effects."
)
print()
print("=" * 70)
print("Figures saved: eda_overview.png, sample_paths_visualization.png, explainability_report.png")

In [ ]:
section_divider("What's Next")

## What's Next

Your project is complete when:

- [ ] Section 1 uses your chosen dataset (not French Bakery)
- [ ] `H` is set to match your planning horizon
- [ ] 80% coverage is between 75% and 85%
- [ ] Business question is answered with a single numeric result
- [ ] Top feature is interpreted in your domain's language
- [ ] Executive summary paragraph reads cleanly to a non-technical colleague

Run `exercises/01_milestone_checklist.py` to verify all deliverables programmatically.

---

**Potential next steps after this project:**

- Hierarchical foreciliation with `hierarchicalforecast` (aggregate store → region → national)
- Foundation model fine-tuning with Nixtla TimeGPT
- Real-time inference API wrapping the trained `NeuralForecast` object
- Calibration drift monitoring on rolling windows in production

In [ ]:
key_takeaways([
    "Review the key concepts covered in this notebook",
    "Practice applying these techniques to your own data",
    "Connect this material to the companion guide and slides"
])